# Re-analyze GuardBench effectiveness from saved predictions

This notebook recomputes metrics from `{out_dir}/{dataset_alias}/{model_name}.json` files **without running inference**. It uses:

- `guardbench.evaluate.evaluate` — same thresholded metrics as `benchmark()`
- `guardbench.report.Report` — optional comparison tables across models (expects prediction JSON to cover every test-split id)

The first cell sets **ROOT** by (1) walking **up** from Jupyter’s cwd to a directory containing `guardbench/__init__.py`, then (2) if that fails, using the **`guardbench` install location** when it looks like an editable checkout (paths under `site-packages` are ignored).


Your **venv does not choose cwd**. If ROOT is still wrong or you used a custom `--out-dir`, set **OUT_DIR** explicitly in the next cell.


In [5]:
from pathlib import Path

# Jupyter's cwd is often NOT the repo root (e.g. notebooks/, project root in IDEs).
# Walk upward from cwd to find GuardBench instead of relying on os.getcwd().
import os

def find_repo_root(start: Path | None = None) -> Path | None:
    cur = Path(start or os.getcwd()).resolve()
    for p in [cur, *cur.parents]:
        pkg = p / "guardbench" / "__init__.py"
        if pkg.is_file():
            return p
    return None


def infer_repo_root_from_installed_package() -> Path | None:
    """If GuardBench was installed editable from a checkout, infer repo root."""
    import importlib.util

    spec = importlib.util.find_spec("guardbench")
    if spec is None or not getattr(spec, "origin", None):
        return None
    init_py = Path(spec.origin).resolve()
    pkg_dir = init_py.parent
    if pkg_dir.name != "guardbench":
        return None
    candidate = pkg_dir.parent
    # Wheels live under site-packages — not a usable project root here
    if "site-packages" in candidate.parts:
        return None
    return candidate


_repo_root = find_repo_root() or infer_repo_root_from_installed_package()
ROOT = _repo_root if _repo_root is not None else Path.cwd()
if _repo_root is None:
    print(
        "Note: Could not find guardbench/__init__.py above cwd.\n"
        "Using cwd as ROOT — set OUT_DIR in the next cell if results live elsewhere.\n"
        f" cwd = {Path.cwd()}",
    )

from unified_io import read_json

from guardbench.datasets import DATASETS, load_dataset
from guardbench.evaluate import evaluate
from guardbench.report import Report

In [6]:
# -----------------------------
# Paths and analysis parameters
# -----------------------------

OUT_DIR = ROOT / "results"  # same default as gspr_vllm.py

# Dataset aliases — same strings as benchmarks (e.g. harmbench_behaviors)
DATASET_ALIASES = [
    "jbb_behaviors",
    "xstest",
    "harmbench_behaviors",
    "advbench_behaviors",
    "advbench_strings",
]

# JSON stem(s) — must match filenames: OUT_DIR /<alias>/<stem>.json
MODEL_STEMS = ["GSPR", "GPT-OSS-Safeguard-20B", "LlamaGuard3-8B", "Qwen3Guard"]

# Metrics printed in the per-dataset table (same naming as benchmarking)
DISPLAY_METRICS = ["precision", "recall", "f1", "auprc"]

# Probability threshold for unsafe vs safe (benchmark uses 0.5)
THRESHOLD = 0.5

# If True, subset ground truth to the ids present in the prediction file.
# Use True when the benchmark was run with max_examples / smoke-test;
# use False for full-test runs matching the dataset test split exactly.
ALIGN_GROUND_TRUTH_TO_PREDICTION_IDS = True

## Helper: reload labels + preds and call `evaluate`

`Report` expects the full test split to match prediction keys exactly. For partial runs, we restrict `y_true` to prediction ids.

In [7]:
def load_predictions(out_dir: Path, dataset_alias: str, model_stem: str) -> dict:
    path = out_dir / dataset_alias / f"{model_stem}.json"
    return read_json(path)


def ground_truth_for_eval(
    dataset_alias: str,
    y_pred_prob: dict,
    *,
    split: str = "test",
    align_to_predictions: bool,
) -> dict:
    ds = load_dataset(dataset_alias)
    if ds.current_split != split:
        ds.load(split)
    y_full = ds.get_ground_truth(split)

    pred_ids = set(y_pred_prob)
    gt_ids = set(y_full)

    if align_to_predictions:
        missing_gt = pred_ids - gt_ids
        if missing_gt:
            raise ValueError(
                f"Prediction IDs not found in dataset {dataset_alias}: "
                f"{sorted(missing_gt)[:5]}..."
            )
        return {pid: y_full[pid] for pid in pred_ids}

    if pred_ids != gt_ids:
        raise ValueError(
            "Ground truth ids != prediction ids. "
            "Set ALIGN_GROUND_TRUTH_TO_PREDICTION_IDS = True "
            "or re-run benchmarking on the full test split.\n"
            f"Only in preds: {sorted(pred_ids - gt_ids)[:8]} …\n"
            f"Only in GT: {sorted(gt_ids - pred_ids)[:8]} …"
        )
    return y_full


def evaluate_saved_run(
    out_dir: Path,
    dataset_alias: str,
    model_stem: str,
    *,
    threshold: float = 0.5,
    align_to_predictions: bool = True,
) -> dict:
    """Mirror of benchmark()'s evaluate step using files on disk."""
    y_pred_prob = load_predictions(out_dir, dataset_alias, model_stem)
    y_true = ground_truth_for_eval(
        dataset_alias,
        y_pred_prob,
        align_to_predictions=align_to_predictions,
    )
    return evaluate(y_true, y_pred_prob, threshold=threshold)

## Results wide table (`evaluate`, same metrics as benchmarking)

Models as **rows**. Each **dataset** is a column group with Precision, Recall, F1, and AUPRC (from `DISPLAY_METRICS`).


In [8]:
import pandas as pd
from guardbench.benchmark.effectiveness import metric_mapping

try:
    from IPython.display import display as ipython_display
except ImportError:
    ipython_display = None

_reports: dict[str, dict[str, dict]] = {}
for stem in MODEL_STEMS:
    _reports[stem] = {}
    for ds_alias in DATASET_ALIASES:
        _reports[stem][ds_alias] = evaluate_saved_run(
            OUT_DIR,
            ds_alias,
            stem,
            threshold=THRESHOLD,
            align_to_predictions=ALIGN_GROUND_TRUTH_TO_PREDICTION_IDS,
        )

_dataset_labels = [DATASETS.get(a, a) for a in DATASET_ALIASES]
_columns = pd.MultiIndex.from_tuples(
    [
        (label, metric_mapping[m])
        for label in _dataset_labels
        for m in DISPLAY_METRICS
    ],
    names=("Dataset", "Metric"),
)

_wide = pd.DataFrame(
    [
        [_reports[stem][alias][m] for alias in DATASET_ALIASES for m in DISPLAY_METRICS]
        for stem in MODEL_STEMS
    ],
    index=pd.Index(MODEL_STEMS, name="Model"),
    columns=_columns,
)

# Plain DataFrame (pandas .style requires jinja2 — not used here)
_wide_display = _wide.round(3)

if ipython_display is not None:
    ipython_display(_wide_display)
else:
    print(_wide_display.to_string())


Dataset               JBB Behaviors                         XSTest         \
Metric                    Precision Recall     F1  AUPRC Precision Recall   
Model                                                                       
GSPR                          0.769   1.00  0.870  0.769     0.910  0.955   
GPT-OSS-Safeguard-20B         0.725   1.00  0.840  0.725     0.853  0.960   
LlamaGuard3-8B                0.865   0.96  0.910  0.850     0.959  0.820   
Qwen3Guard                    0.812   0.95  0.876  0.802     0.982  0.815   

Dataset                             HarmBench Behaviors                    \
Metric                    F1  AUPRC           Precision Recall   F1 AUPRC   
Model                                                                       
GSPR                   0.932  0.889                 0.0  0.794  0.0   0.0   
GPT-OSS-Safeguard-20B  0.904  0.843                 0.0  0.994  0.0   0.0   
LlamaGuard3-8B         0.884  0.866                 0.0  0.934  0.0   0.0   
Qwen3Guard             0.891  0.948                 0.0  0.938  0.0   0.0   

Dataset               AdvBench Behaviors                   AdvBench Strings  \
Metric                         Precision Recall   F1 AUPRC        Precision   
Model                                                                         
GSPR                                 0.0  0.990  0.0   0.0              0.0   
GPT-OSS-Safeguard-20B                0.0  0.998  0.0   0.0              0.0   
LlamaGuard3-8B                       0.0  0.975  0.0   0.0              0.0   
Qwen3Guard                           0.0  0.994  0.0   0.0              0.0   

Dataset                                  
Metric                Recall   F1 AUPRC  
Model                                    
GSPR                   0.925  0.0   0.0  
GPT-OSS-Safeguard-20B  0.883  0.0   0.0  
LlamaGuard3-8B         0.800  0.0   0.0  
Qwen3Guard             0.817  0.0   0.0